# Assignment 3 DSC 208R WI24

## Introduction

In this assignment we will conduct data engineering for the Amazon dataset. It is divided into 2 parts. The extracted features in Part 1 will be used for the Part 2 of assignment, where you train a model (or models) to predict user ratings for a product.

We will be using Apache Spark for this assignment. The default Spark API will be DataFrame, as it is now the recommended choice over the RDD API. That being said, please feel free to switch back to the RDD API if you see it as a better fit for the task. We provide you an option to request RDD format to start with. Also you can switch between DataFrame and RDD in your solution. 

Another newer API is Koalas, which is also avaliable. However, it has constraints and is not applicable to most tasks. Refer to the PA statement for detail.

### Set the following parameters

In [ ]:
PID = 'a18487024' # your pid, for instance: 'a43223333'
INPUT_FORMAT = 'dataframe' # choose a format of your input data, valid options: 'dataframe', 'rdd', 'koalas'

In [ ]:
# Boiler plates, do NOT modify
%load_ext autoreload
%autoreload 2
import os
import getpass
from pyspark.sql import SparkSession
from utilities import SEED
from utilities import PA3Test
from utilities import PA3Data
from utilities import data_cat
from pa3_main import PA3Executor
import time
if INPUT_FORMAT == 'dataframe':
    import pyspark.ml as M
    import pyspark.sql.functions as F
    import pyspark.sql.types as T
if INPUT_FORMAT == 'koalas':
    import databricks.koalas as ks
elif INPUT_FORMAT == 'rdd':
    import pyspark.mllib as M
    from pyspark.mllib.feature import Word2Vec
    from pyspark.mllib.linalg import Vectors
    from pyspark.mllib.linalg.distributed import RowMatrix

os.environ['PYSPARK_SUBMIT_ARGS'] = '--py-files utilities.py,assignment3.py \
--deploy-mode client \
pyspark-shell'

class args:
    review_filename = data_cat.review_filename
    product_filename = data_cat.product_filename
    product_processed_filename = data_cat.product_processed_filename
    ml_features_train_filename = data_cat.ml_features_train_filename
    ml_features_test_filename = data_cat.ml_features_test_filename
    output_root = '/home/{}/{}-pa3/test_results'.format(getpass.getuser(), PID)
    test_results_root = data_cat.test_results_root
    pid = PID

pa3 = PA3Executor(args, input_format=INPUT_FORMAT)
data_io = pa3.data_io
data_dict = pa3.data_dict
begin = time.time()


26/05/30 03:18:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/bitnami/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Loading datasets ...

Done


In [ ]:
# Import your own dependencies
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

# Set log level to WARN or ERROR to reduce console clutter
sc.setLogLevel("ERROR")

import pyspark.sql.functions as F
import pyspark.sql.types as T
import numpy as np
import pandas as pd

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier



In [ ]:
data_dict['review'] = data_dict['review'].coalesce(8)
data_dict['product'] = data_dict['product'].coalesce(8)
data_dict['product_processed'] = data_dict['product_processed'].coalesce(8)
data_dict['ml_features_train'] = data_dict['ml_features_train'].coalesce(8)
data_dict['ml_features_test'] = data_dict['ml_features_test'].coalesce(8)


# Part 1: Feature Engineering

In [ ]:
# Bring the part_1 datasets to memory and de-cache part_2 datasets. 
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_1')

In [ ]:
pa3.spark


In [ ]:
for name, df in data_dict.items():
    print(name, df.is_cached)


review False
product False
product_processed False
ml_features_train False
ml_features_test False


# Task0: warm up 
This task is provided for you to get familiar with Spark API. We will use the dataframe API to demonstrate. Solution is given to you and this task won't be graded.

Refer to https://spark.apache.org/docs/latest/api/python/pyspark.sql.html for API guide.

The task is to implement the function below. Given the ```product_data``` table:
1. Take and print five rows.

1. Select only the ```asin``` column, then print five rows of it.

1. Select the row where ```asin = B00I8KEOTM``` and print it.

1. Count the total number of rows.

1. Calculate the mean ```price```.

1. You need to conduct the above operations, then extract some statistics out of the generated columns. You need to put the statistics in a python dictionary named ```res```. The description and schema of it are as follows:
    ```
    res
     | -- count_total: int -- count of total rows of the entire table after your operations
     | -- mean_price: float -- mean value of column price
    ```

In [ ]:
def task_0(data_io, product_data):
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    overall_column = 'overall'
    # Outputs:
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------

    product_data.show(5)
    product_data[['asin']].show(5)
    product_data.where(F.col('asin') == 'B00I8KEOTM').show()
    count_rows = product_data.count()
    mean_price = product_data.select(F.avg(F.col('price'))).head()[0]
    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    # Calculate the values programmatically. Do not change the keys and do not
    # hard-code values in the dict. Your submission will be evaluated with
    # different inputs.
    # Modify the values of the following dictionary accordingly.
    res = {'count_total': None, 'mean_price': None}
    
    # Modify res:

    res['count_total'] = count_rows
    res['mean_price'] = mean_price

    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    return res
    # -------------------------------------------------------------------------

In [ ]:
if INPUT_FORMAT == 'dataframe':
    res = task_0(data_io, data_dict['product'])
    pa3.tests.test(res, 'task_0')


[Stage 18:>                                                         (0 + 1) / 1]



+----------+--------------------+--------------------+--------------------+-----+--------------------+
|      asin|           salesRank|          categories|               title|price|             related|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
|B00HCE65S8|                null|[[Apps for Androi...|                null| null|{also_bought -> [...|
|B00HCE9GLQ|{Electronics -> 6...| [[All Electronics]]|Sony SRF-M37 Walk...|33.74|{also_viewed -> [...|
|B00HCDG32C|                null|            [[Baby]]|Qubies Baby Food ...|19.99|{also_bought -> [...|
|B00HCDCVIC|{Clothing -> 478635}|[[Clothing, Shoes...|London Times Wome...|70.54|{also_viewed -> [...|
|B00HCDE38S|{Clothing -> 149857}|[[Clothing, Shoes...|London Times Wome...|66.22|{also_viewed -> [...|
+----------+--------------------+--------------------+--------------------+-----+--------------------+
only showing top 5 rows

+----------+
|      asin|
+----------+
|B00HCE65

+----------+---------+--------------------+-----+-----+--------------------+
|      asin|salesRank|          categories|title|price|             related|
+----------+---------+--------------------+-----+-----+--------------------+
|B00I8KEOTM|     null|[[Apps for Androi...| null| null|{also_viewed -> [...|
+----------+---------+--------------------+-----+-----+--------------------+



tests for task_0 --------------------------------------------------------------
2/2 passed
-------------------------------------------------------------------------------


# Task1

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()



In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", 8)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)


In [ ]:
# %load -s task_1 assignment3.py
def task_1(data_io, review_data, product_data):
    from pyspark.sql import functions as F

    # ----------------------------- Column names -----------------------------
    asin_column = 'asin'
    overall_column = 'overall'
    mean_rating_column = 'meanRating'
    count_rating_column = 'countRating'
    # ------------------------------------------------------------------------
    
    
    
    # ---------------------- Your implementation begins------------------------
    # 1 Aggregate review data per product
    review_stats = (
        review_data
        .groupBy(asin_column)
        .agg(
            F.mean(overall_column).alias(mean_rating_column),
            F.count(overall_column).alias(count_rating_column)
        )
    )

    # 2 Join with product_data and cache once
    joined_data = product_data.join(review_stats, on=asin_column, how='left').cache()

    # 3 Compute all metrics in one aggregation call
    stats = joined_data.agg(
        F.count('*').alias('count_total'),
        F.mean(mean_rating_column).alias('mean_meanRating'),
        F.variance(mean_rating_column).alias('variance_meanRating'),
        F.mean(count_rating_column).alias('mean_countRating'),
        F.variance(count_rating_column).alias('variance_countRating')
    ).collect()[0]

    # 4 Compute null counts separately (lightweight filters)
    numNulls_meanRating = joined_data.filter(F.col(mean_rating_column).isNull()).count()
    numNulls_countRating = joined_data.filter(F.col(count_rating_column).isNull()).count()

    # 5 Build result dictionary
    res = {
        'count_total': stats['count_total'],
        'mean_meanRating': stats['mean_meanRating'],
        'variance_meanRating': stats['variance_meanRating'],
        'numNulls_meanRating': numNulls_meanRating,
        'mean_countRating': stats['mean_countRating'],
        'variance_countRating': stats['variance_countRating'],
        'numNulls_countRating': numNulls_countRating
    }

    # 6 Save and return
    data_io.save(res, 'task_1')
    return res


In [ ]:
res = task_1(data_io, data_dict['review'], data_dict['product'])
pa3.tests.test(res, 'task_1')


# Task 2

In [ ]:
# %load -s task_2 assignment3.py
def task_2(data_io, product_data):
    from pyspark.sql import functions as F
    # -----------------------------Column names--------------------------------
    # Inputs:
    salesRank_column = 'salesRank'
    categories_column = 'categories'
    asin_column = 'asin'
    # Outputs:
    category_column = 'category'
    bestSalesCategory_column = 'bestSalesCategory'
    bestSalesRank_column = 'bestSalesRank'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------

    # 1 Extract category and best sales category
    # Each product may have multiple categories; take the first one as 'category'
    product_data = product_data.withColumn(category_column, F.element_at(F.col(categories_column), 1))

    # 2 Extract best sales category and rank from salesRank map
    # salesRank is a map<string,int>; find the key with the smallest rank value
    product_data = product_data.withColumn(bestSalesCategory_column, F.map_keys(F.col(salesRank_column))[0])
    product_data = product_data.withColumn(bestSalesRank_column, F.map_values(F.col(salesRank_column))[0])

    # 3 Compute overall statistics
    count_total = product_data.count()
    mean_bestSalesRank = product_data.select(F.mean(bestSalesRank_column)).head()[0]
    variance_bestSalesRank = product_data.select(F.variance(bestSalesRank_column)).head()[0]

    numNulls_category = product_data.filter(F.col(category_column).isNull()).count()
    countDistinct_category = product_data.select(category_column).distinct().count()

    numNulls_bestSalesCategory = product_data.filter(F.col(bestSalesCategory_column).isNull()).count()
    countDistinct_bestSalesCategory = product_data.select(bestSalesCategory_column).distinct().count()



    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    res = {
        'count_total': count_total,
        'mean_bestSalesRank': mean_bestSalesRank,
        'variance_bestSalesRank': variance_bestSalesRank,
        'numNulls_category': numNulls_category,
        'countDistinct_category': countDistinct_category,
        'numNulls_bestSalesCategory': numNulls_bestSalesCategory,
        'countDistinct_bestSalesCategory': countDistinct_bestSalesCategory
    }
    # Modify res:




    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_2')
    return res
    # -------------------------------------------------------------------------


In [ ]:
res = task_2(data_io, data_dict['product'])
pa3.tests.test(res, 'task_2')

NameError: name 'data_io' is not defined

# Task 3





In [ ]:
# %load -s task_3 assignment3.py
def task_3(data_io, product_data):
    from pyspark.sql import functions as F
    # -----------------------------Column names--------------------------------
    # Inputs:
    asin_column = 'asin'
    price_column = 'price'
    attribute = 'also_viewed'
    related_column = 'related'
    # Outputs:
    meanPriceAlsoViewed_column = 'meanPriceAlsoViewed'
    countAlsoViewed_column = 'countAlsoViewed'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------
    # 1 Extract the 'also_viewed' list from the 'related' map
    product_data = product_data.withColumn(attribute, F.col(f"{related_column}.{attribute}"))

    # 2 Compute count of also_viewed items per product
    product_data = product_data.withColumn(countAlsoViewed_column, F.size(F.col(attribute)))

    # 3 Compute mean price of also_viewed products
    #    Join product_data with itself to get prices of related products
    exploded = product_data.select(
        F.col(asin_column).alias('asin_main'),
        F.explode_outer(F.col(attribute)).alias('asin_related')
    )

    joined = exploded.join(
        product_data.select(asin_column, price_column),
        exploded.asin_related == product_data.asin,
        how='left'
    )

    mean_price_df = (
        joined.groupBy('asin_main')
        .agg(F.mean(price_column).alias(meanPriceAlsoViewed_column))
    )

    # 4 Merge mean price and count columns back into product_data
    product_data = (
        product_data.join(mean_price_df, product_data.asin == mean_price_df.asin_main, how='left')
        .drop('asin_main')
    )

    # 5 Compute overall statistics
    count_total = product_data.count()
    mean_meanPriceAlsoViewed = product_data.select(F.mean(meanPriceAlsoViewed_column)).head()[0]
    variance_meanPriceAlsoViewed = product_data.select(F.variance(meanPriceAlsoViewed_column)).head()[0]
    numNulls_meanPriceAlsoViewed = product_data.filter(F.col(meanPriceAlsoViewed_column).isNull()).count()

    mean_countAlsoViewed = product_data.select(F.mean(countAlsoViewed_column)).head()[0]
    variance_countAlsoViewed = product_data.select(F.variance(countAlsoViewed_column)).head()[0]
    numNulls_countAlsoViewed = product_data.filter(F.col(countAlsoViewed_column).isNull()).count()




    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
  
    res = {
        'count_total': count_total,
        'mean_meanPriceAlsoViewed': mean_meanPriceAlsoViewed,
        'variance_meanPriceAlsoViewed': variance_meanPriceAlsoViewed,
        'numNulls_meanPriceAlsoViewed': numNulls_meanPriceAlsoViewed,
        'mean_countAlsoViewed': mean_countAlsoViewed,
        'variance_countAlsoViewed': variance_countAlsoViewed,
        'numNulls_countAlsoViewed': numNulls_countAlsoViewed
    }
    # Modify res:




    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_3')
    return res
    # -------------------------------------------------------------------------


In [ ]:
res = task_3(data_io, data_dict['product'])
pa3.tests.test(res, 'task_3')

# Task 4

In [ ]:
# %load -s task_4 assignment3.py
def task_4(data_io, product_data):
    from pyspark.sql import functions as F
    # -----------------------------Column names--------------------------------
    # Inputs:
    price_column = 'price'
    title_column = 'title'
    # Outputs:
    meanImputedPrice_column = 'meanImputedPrice'
    medianImputedPrice_column = 'medianImputedPrice'
    unknownImputedTitle_column = 'unknownImputedTitle'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------

    # 1 Compute mean and median of price (excluding nulls)
    mean_price = product_data.select(F.mean(price_column)).head()[0]
    median_price = (
        product_data.approxQuantile(price_column, [0.5], 0.01)[0]
        if product_data.filter(F.col(price_column).isNotNull()).count() > 0
        else None
    )

    # 2 Impute missing prices with mean and median
    product_data = product_data.withColumn(
        meanImputedPrice_column,
        F.when(F.col(price_column).isNull(), mean_price).otherwise(F.col(price_column))
    )

    product_data = product_data.withColumn(
        medianImputedPrice_column,
        F.when(F.col(price_column).isNull(), median_price).otherwise(F.col(price_column))
    )

    # 3 Impute missing titles with placeholder
    product_data = product_data.withColumn(
        unknownImputedTitle_column,
        F.when(F.col(title_column).isNull(), F.lit("Unknown")).otherwise(F.col(title_column))
    )

    # 4 Compute overall statistics
    count_total = product_data.count()

    # Mean-imputed price stats
    mean_meanImputedPrice = product_data.select(F.mean(meanImputedPrice_column)).head()[0]
    variance_meanImputedPrice = product_data.select(F.variance(meanImputedPrice_column)).head()[0]
    numNulls_meanImputedPrice = product_data.filter(F.col(meanImputedPrice_column).isNull()).count()

    # Median-imputed price stats
    mean_medianImputedPrice = product_data.select(F.mean(medianImputedPrice_column)).head()[0]
    variance_medianImputedPrice = product_data.select(F.variance(medianImputedPrice_column)).head()[0]
    numNulls_medianImputedPrice = product_data.filter(F.col(medianImputedPrice_column).isNull()).count()

    # Unknown title count
    numUnknowns_unknownImputedTitle = product_data.filter(
        F.col(unknownImputedTitle_column) == "Unknown"
    ).count()



    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
   
    res = {
        'count_total': count_total,
        'mean_meanImputedPrice': mean_meanImputedPrice,
        'variance_meanImputedPrice': variance_meanImputedPrice,
        'numNulls_meanImputedPrice': numNulls_meanImputedPrice,
        'mean_medianImputedPrice': mean_medianImputedPrice,
        'variance_medianImputedPrice': variance_medianImputedPrice,
        'numNulls_medianImputedPrice': numNulls_medianImputedPrice,
        'numUnknowns_unknownImputedTitle': numUnknowns_unknownImputedTitle
    }
    # Modify res:




    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_4')
    return res
    # -------------------------------------------------------------------------


In [ ]:
res = task_4(data_io, data_dict['product'])
pa3.tests.test(res, 'task_4')

# Task 5

In [ ]:
# %load -s task_5 assignment3.py
def task_5(data_io, product_processed_data, word_0, word_1, word_2):
    from pyspark.sql import functions as F
    from pyspark.ml.feature import Tokenizer, Word2Vec
    # -----------------------------Column names--------------------------------
    # Inputs:
    title_column = 'title'
    # Outputs:
    titleArray_column = 'titleArray'
    titleVector_column = 'titleVector'
    # -------------------------------------------------------------------------

    # ---------------------- Your implementation begins------------------------

    # 1 Tokenize product titles into arrays of words
    tokenizer = Tokenizer(inputCol=title_column, outputCol=titleArray_column)
    tokenized_data = tokenizer.transform(product_processed_data)

    # 2 Train Word2Vec model on tokenized titles
    word2vec = Word2Vec(
        vectorSize=50,
        minCount=2,
        inputCol=titleArray_column,
        outputCol=titleVector_column
    )
    model = word2vec.fit(tokenized_data)
    transformed_data = model.transform(tokenized_data)

    # 3 Compute vocabulary size and total count
    count_total = transformed_data.count()
    size_vocabulary = len(model.getVectors().collect())

    # 4 Find synonyms for the three input words
    synonyms_0 = [(row.word, float(row.similarity)) for row in model.findSynonyms(word_0, 5).collect()]
    synonyms_1 = [(row.word, float(row.similarity)) for row in model.findSynonyms(word_1, 5).collect()]
    synonyms_2 = [(row.word, float(row.similarity)) for row in model.findSynonyms(word_2, 5).collect()]



    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
    
    res = {
        'count_total': count_total,
        'size_vocabulary': size_vocabulary,
        'word_0_synonyms': synonyms_0,
        'word_1_synonyms': synonyms_1,
        'word_2_synonyms': synonyms_2
    }
    # Modify res:
    
    
    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_5')
    return res
    # -------------------------------------------------------------------------


In [ ]:
res = task_5(data_io, data_dict['product_processed'], 'piano', 'rice', 'laptop')
pa3.tests.test(res, 'task_5')

# Task 6

In [ ]:
# %load -s task_6 assignment3.py
def task_6(data_io, product_processed_data):
    from pyspark.sql import functions as F
    from pyspark.ml.feature import StringIndexer, OneHotEncoder, PCA
    from pyspark.ml.linalg import Vectors
    
    # -----------------------------Column names--------------------------------
    # Inputs:
    category_column = 'category'
    # Outputs:
    categoryIndex_column = 'categoryIndex'
    categoryOneHot_column = 'categoryOneHot'
    categoryPCA_column = 'categoryPCA'
    # -------------------------------------------------------------------------    

    # ---------------------- Your implementation begins------------------------

    # 1 Convert category strings to numeric indices
    indexer = StringIndexer(inputCol=category_column, outputCol=categoryIndex_column)
    indexed_data = indexer.fit(product_processed_data).transform(product_processed_data)

    # 2 Apply one-hot encoding
    encoder = OneHotEncoder(inputCols=[categoryIndex_column], outputCols=[categoryOneHot_column])
    encoded_data = encoder.fit(indexed_data).transform(indexed_data)

    # 3 Apply PCA to reduce dimensionality of one-hot vectors
    pca = PCA(k=3, inputCol=categoryOneHot_column, outputCol=categoryPCA_column)
    pca_model = pca.fit(encoded_data)
    pca_data = pca_model.transform(encoded_data)

    # 4 Compute summary statistics
    count_total = pca_data.count()

    # Mean vector of one-hot encoded categories
    meanVector_categoryOneHot = (
        pca_data.select(F.mean(categoryOneHot_column)).head()[0].toArray().tolist()
        if count_total > 0 else []
    )

    # Mean vector of PCA-reduced categories
    meanVector_categoryPCA = (
        pca_data.select(F.mean(categoryPCA_column)).head()[0].toArray().tolist()
        if count_total > 0 else []
    )



    # -------------------------------------------------------------------------

    # ---------------------- Put results in res dict --------------------------
   
    res = {
        'count_total': count_total,
        'meanVector_categoryOneHot': meanVector_categoryOneHot,
        'meanVector_categoryPCA': meanVector_categoryPCA
    }
    # Modify res:




    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_6')
    return res
    # -------------------------------------------------------------------------


In [ ]:
res = task_6(data_io, data_dict['product_processed'])
pa3.tests.test(res, 'task_6')

In [ ]:
print ("End to end time: {}".format(time.time()-begin))

# Part 2: Model Selection

In [ ]:
# Bring the part_2 datasets to memory and de-cache part_1 datasets.
# Execute this once before you start working on this Part
data_dict, _ = data_io.cache_switch(data_dict, 'part_2')

# Task 7

In [ ]:
def task_7(data_io, train_data, test_data):
    from pyspark.ml.regression import DecisionTreeRegressor
    from pyspark.ml.evaluation import RegressionEvaluator

    
    # ---------------------- Your implementation begins------------------------
    
    # Train model
    dt = DecisionTreeRegressor(featuresCol="features", labelCol="rating", maxDepth=5)
    model = dt.fit(train_data)

    # Predict on test data
    predictions = model.transform(test_data)

    # Evaluate RMSE
    evaluator = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="rmse")
    test_rmse = evaluator.evaluate(predictions)
    
    
    
    # -------------------------------------------------------------------------
    
    
    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rmse': test_rmse
    }
    # Modify res:


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_7')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_7(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa3.tests.test(res, 'task_7')

# Task 8

In [ ]:
def task_8(data_io, train_data, test_data):
    from pyspark.ml.regression import DecisionTreeRegressor
    from pyspark.ml.evaluation import RegressionEvaluator
    
    # ---------------------- Your implementation begins------------------------
    # 1 Split original training data into 75% train, 25% validation
    train_split, valid_split = train_data.randomSplit([0.75, 0.25], seed=42)

    depths = [5, 7, 9, 12]
    valid_rmses = {}

    # 2 Train models with different depths and evaluate on validation set
    for depth in depths:
        dt = DecisionTreeRegressor(featuresCol="features", labelCol="rating", maxDepth=depth)
        model = dt.fit(train_split)
        preds = model.transform(valid_split)
        evaluator = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="rmse")
        rmse = evaluator.evaluate(preds)
        valid_rmses[depth] = rmse

    # 3 Select best depth (lowest validation RMSE)
    best_depth = min(valid_rmses, key=valid_rmses.get)

    # 4 Retrain on full training data using best depth and evaluate on test data
    best_dt = DecisionTreeRegressor(featuresCol="features", labelCol="rating", maxDepth=best_depth)
    best_model = best_dt.fit(train_data)
    test_preds = best_model.transform(test_data)
    test_rmse = evaluator.evaluate(test_preds)
    
    
    
    
    # -------------------------------------------------------------------------
    
    
    # ---------------------- Put results in res dict --------------------------
    res = {
        'test_rms': test_rmse,
        'valid_rms_depth_5': valid_rmses[5],
        'valid_rms_depth_7': valid_rmses[7],
        'valid_rms_depth_9': valid_rmses[9],
        'valid_rms_depth_12': valid_rmses[12],
    }

    # Modify res:


    # -------------------------------------------------------------------------

    # ----------------------------- Do not change -----------------------------
    data_io.save(res, 'task_8')
    return res
    # -------------------------------------------------------------------------

In [ ]:
res = task_8(data_io, data_dict['ml_features_train'], data_dict['ml_features_test'])
pa3.tests.test(res, 'task_8')

In [ ]:
print ("End to end time: {}".format(time.time()-begin))